In [ ]:
YEAR_FILTER = '0113-1'  # 修改為你想篩選的年份

In [104]:
import pandas as pd
import numpy as np
import os

YEAR_FILTER = '0113-1'  # 修改為你想篩選的年份

INPUT_FOLDER = f'processed_data_{YEAR_FILTER}/'  # 資料夾路徑
OUTPUT_FOLDER = f'processed_courses_{YEAR_FILTER}/'  # 新建的課程檔案路徑
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def process_files(input_folder, output_folder):
    """逐一處理每個 113-1.csv 檔案，提取課程名稱並找到對應資料"""
    files = os.listdir(input_folder)
    
    # 找到 113-1 和 other_years 的檔案對應
    for file in files:
        if file.endswith(f'{YEAR_FILTER}.csv'):
            # 提取對應的 other_years.csv 檔案名稱
            department = file.split('_')[0]
            other_years_file = f"{department}_other_years.csv"

            if other_years_file in files:
                # 讀取 113-1.csv 和 other_years.csv
                main_df = pd.read_csv(os.path.join(input_folder, file))
                other_df = pd.read_csv(os.path.join(input_folder, other_years_file))

                if '科目名稱(連結課程地圖)             備註 \n             限選條件' not in main_df.columns or '科目名稱(連結課程地圖)             備註 \n             限選條件' not in other_df.columns:
                    print(f"跳過 {file}，因為科目名稱欄位不存在")
                    continue

                # 提取 113-1.csv 中的課程名稱
                course_names = main_df['科目名稱(連結課程地圖)             備註 \n             限選條件'].unique()

                # 建立系所資料夾
                department_folder = os.path.join(output_folder, department)
                os.makedirs(department_folder, exist_ok=True)

                # 對於每個課程名稱，找到其他年份的對應資料並儲存
                for course_name in course_names:
                    # 從 113-1.csv 和 other_years.csv 中篩選出對應課程資料
                    main_course_df = main_df[main_df['科目名稱(連結課程地圖)             備註 \n             限選條件'] == course_name]
                    other_course_df = other_df[other_df['科目名稱(連結課程地圖)             備註 \n             限選條件'] == course_name]

                    # 合併資料
                    combined_df = pd.concat([main_course_df, other_course_df], ignore_index=True)

                    # 儲存結果
                    sanitized_course_name = course_name.replace('/', '_').replace('\\', '_').replace(':', '_')
                    output_file = os.path.join(department_folder, f"{sanitized_course_name}.csv")
                    combined_df.to_csv(output_file, index=False, encoding='utf-8')
                    print(f"已儲存課程 '{course_name}' 的結合結果至 {output_file}")
            else:
                print(f"找不到 {department} 的對應檔案 {other_years_file}")

if __name__ == '__main__':
    process_files(INPUT_FOLDER, OUTPUT_FOLDER)

已儲存課程 '微積分（一）' 的結合結果至 processed_courses_0113-1/數學系 MATH\微積分（一）.csv
已儲存課程 '數學導論' 的結合結果至 processed_courses_0113-1/數學系 MATH\數學導論.csv
已儲存課程 '普通物理學（一）' 的結合結果至 processed_courses_0113-1/數學系 MATH\普通物理學（一）.csv
已儲存課程 '線性代數（一）' 的結合結果至 processed_courses_0113-1/數學系 MATH\線性代數（一）.csv
已儲存課程 '高等微積分（一）' 的結合結果至 processed_courses_0113-1/數學系 MATH\高等微積分（一）.csv
已儲存課程 '代數學（一）' 的結合結果至 processed_courses_0113-1/數學系 MATH\代數學（一）.csv
已儲存課程 '機率導論' 的結合結果至 processed_courses_0113-1/數學系 MATH\機率導論.csv
已儲存課程 '數論導論' 的結合結果至 processed_courses_0113-1/數學系 MATH\數論導論.csv
已儲存課程 '向量分析' 的結合結果至 processed_courses_0113-1/數學系 MATH\向量分析.csv
已儲存課程 '初等分析' 的結合結果至 processed_courses_0113-1/數學系 MATH\初等分析.csv
已儲存課程 '複變數函數論' 的結合結果至 processed_courses_0113-1/數學系 MATH\複變數函數論.csv
已儲存課程 '數值分析（一）' 的結合結果至 processed_courses_0113-1/數學系 MATH\數值分析（一）.csv
已儲存課程 '動態系統' 的結合結果至 processed_courses_0113-1/數學系 MATH\動態系統.csv
已儲存課程 '偏微分方程導論' 的結合結果至 processed_courses_0113-1/數學系 MATH\偏微分方程導論.csv
已儲存課程 '拓樸學' 的結合結果至 processed_courses_0113-1/數學系 MATH\拓樸學.csv
已儲存課程 '幾何學（

In [ ]:
def split_enrollment_and_balance(df):
    """將選課人數/餘額欄位分割成兩個欄位"""
    if '已選課人數/餘額' in df.columns:
        df[['選課人數', '餘額']] = df['已選課人數/餘額'].str.split('/', expand=True)
        # 將選課人數和餘額轉換為數字，並補空白為 0
        df['選課人數'] = pd.to_numeric(df['選課人數'], errors='coerce').fillna(0).astype(int)
        df['餘額'] = pd.to_numeric(df['餘額'], errors='coerce').fillna(0).astype(int)
        df.drop('已選課人數/餘額', axis=1, inplace=True)  # 刪除原始欄位
    return df

def weighted_average_predict(df, target_column, year_column, predict_year):
    """依照學年度進行加權平均預測，排除指定學年度的資料"""
    if target_column not in df.columns or year_column not in df.columns:
        print("缺少必要欄位，無法進行加權平均")
        return None

    # 排除 PREDICT_YEAR 的資料
    filtered_df = df[df[year_column] != predict_year].copy()
    course_name = df['科目名稱(連結課程地圖)             備註 \n             限選條件'].iloc[0].replace('\n', ' ')
    if len(filtered_df) < 2:
        print(f"資料不足（少於 2 筆），無法進行加權平均，課程 {course_name} 已被排除")
        return None

    # 計算權重：以學年度的數值大小作為降序順序
    filtered_df['order'] = filtered_df[year_column].rank(method='dense', ascending=False)
    raw_weights = 1 / filtered_df['order']  # 計算初始權重
    filtered_df['標準化權重'] = raw_weights / raw_weights.sum()  # 標準化權重

    # 計算加權平均
    weighted_sum = np.sum(filtered_df['標準化權重'] * filtered_df[target_column])
    total_weight = np.sum(filtered_df['標準化權重'])

    if total_weight == 0:
        print("總權重為零，無法計算加權平均")
        return None

    # 返回四捨五入的結果，保留到小數點後兩位
    return round(weighted_sum, 0)

def process_and_predict(folder_path, output_folder, predict_year):
    """對每個課程進行加權平均預測，並按系儲存結果"""
    for department in os.listdir(folder_path):
        department_folder = os.path.join(folder_path, department)
        if os.path.isdir(department_folder):
            print(f"處理系所: {department}")
            predictions = []

            for file in os.listdir(department_folder):
                if file.endswith('.csv'):  # 僅處理 CSV 檔案
                    file_path = os.path.join(department_folder, file)
                    print(f"讀取課程檔案: {file_path}")
                    
                    # 讀取課程檔案
                    course_df = pd.read_csv(file_path)

                    if '學年期' not in course_df.columns or '已選課人數/餘額' not in course_df.columns:
                        print(f"檔案 {file} 缺少必要欄位，跳過處理")
                        continue

                    # 分割選課人數/餘額
                    course_df = split_enrollment_and_balance(course_df)

                    # 將學年期轉換為整數格式（去掉 '-' 並轉換為數值）
                    course_df['學年期'] = course_df['學年期'].astype(str).str.replace('-', '').astype(int)

                    # 加權平均預測選課人數與餘額
                    predicted_enrollment = weighted_average_predict(course_df, '選課人數', '學年期', predict_year)
                    predicted_balance = weighted_average_predict(course_df, '餘額', '學年期', predict_year)

                    if predicted_enrollment is not None and predicted_balance is not None:
                        # 儲存課程名稱與預測結果
                        course_name = file.replace('.csv', '')
                        predictions.append({
                            '科目名稱': course_name,
                            '預測選課': predicted_enrollment,
                            '預測餘額': predicted_balance
                        })
            
            # 儲存該系所的預測結果為 CSV
            if predictions:
                department_output_file = os.path.join(output_folder, f"{department}_predictions.csv")
                pd.DataFrame(predictions).to_csv(department_output_file, index=False, encoding='utf-8')
                print(f"已儲存 {department} 的預測結果至 {department_output_file}")

PROCESSED_FOLDER = f'processed_courses_{YEAR_FILTER}/'  # 儲存課程的資料夾路徑
OUTPUT_FOLDER = f'predicted_results_{YEAR_FILTER}/'  # 預測結果的輸出資料夾路徑
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if __name__ == '__main__':
    process_and_predict(PROCESSED_FOLDER, OUTPUT_FOLDER, YEAR_FILTER)

處理系所: 數學系 MATH
讀取課程檔案: processed_courses_0113-1/數學系 MATH\代數學（一）.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\偏微分方程導論.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\初等分析.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\動態系統.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\向量分析.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\幾何學（二）.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\微積分（一）.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\拓樸學.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\數值分析（一）.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\數學導論.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\數論導論.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\普通物理學（一）.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\機率導論.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\線性代數（一）.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\複變數函數論.csv
讀取課程檔案: processed_courses_0113-1/數學系 MATH\高等微積分（一）.csv
已儲存 數學系 MATH 的預測結果至 predicted_results_0113-1/數學系 MATH_predictions.csv
處理系所: 測量系 GM
讀取課程檔案: processed_courses_0113-1/測量系 GM\合成孔徑雷達基礎原理與應用.csv
讀取課程

讀取檔案: processed_data_0113-1/數學系 MATH_other_years.csv
讀取檔案: processed_data_0113-1/測量系 GM_other_years.csv
讀取檔案: processed_data_0113-1/統計系 STAT_other_years.csv
讀取檔案: processed_data_0113-1/資訊系 CSIE_other_years.csv
已將包含 'other_years' 的檔案合併並儲存至 combine_data_0113-1/combined_other_years.csv
讀取檔案: processed_data_0113-1/數學系 MATH_0113-1.csv
讀取檔案: processed_data_0113-1/測量系 GM_0113-1.csv
讀取檔案: processed_data_0113-1/統計系 STAT_0113-1.csv
讀取檔案: processed_data_0113-1/資訊系 CSIE_0113-1.csv
已將包含 '0113-1' 的檔案合併並儲存至 combine_data_0113-1/combined_0113-1.csv


One-Hot Encoding 完成，最終特徵數量：518


ValueError: Input y contains NaN.